In [1]:
import pandas as pd
import statsmodels.api as sm

d = pd.read_csv("../data/model_table.csv")
cols = ["TR", "FR", "SR"]

def fit(data):
    X = sm.add_constant(data[cols])
    return sm.Logit(data["Response"], X).fit(disp=0)

s16 = d[d["source"] == "Staley and others (2016)"]
print(len(s16), "rows from the 2016 inventory")

published = pd.Series({"const": -3.63, "TR": 0.41, "FR": 0.67, "SR": 0.70})
compare = pd.DataFrame({
    "published": published,
    "2016 rows": fit(s16).params,
    "all rows": fit(d).params,
    "California": fit(d[d["state"] == "CA"]).params,
})
print(compare.round(2))

1320 rows from the 2016 inventory
       published  2016 rows  all rows  California
const      -3.63      -2.03     -3.25       -2.77
TR          0.41       0.31      0.26        0.34
FR          0.67      -0.07     -0.02        0.18
SR          0.70       0.09      0.34       -0.17


In [2]:
import numpy as np
from sklearn.metrics import roc_auc_score

def m1_p(R, T, F, S, B=-3.63, Ct=0.41, Cf=0.67, Cs=0.70):
    X = B + Ct*T*R + Cf*F*R + Cs*S*R
    return np.exp(X) / (1 + np.exp(X))

for name, data in [("2016 rows", s16), ("all rows", d)]:
    p_pub = m1_p(data["R"], data["T"], data["F"], data["S"])
    p_mine = fit(data).predict(sm.add_constant(data[cols]))
    print(name, "| AUC published:", round(roc_auc_score(data["Response"], p_pub), 3),
          "| AUC my refit:", round(roc_auc_score(data["Response"], p_mine), 3))

print(d[cols].corr().round(2))
print(fit(s16).summary())

2016 rows | AUC published: 0.789 | AUC my refit: 0.777
all rows | AUC published: 0.845 | AUC my refit: 0.859
      TR   FR    SR
TR  1.00  0.8  0.71
FR  0.80  1.0  0.80
SR  0.71  0.8  1.00
                           Logit Regression Results                           
Dep. Variable:               Response   No. Observations:                 1320
Model:                          Logit   Df Residuals:                     1316
Method:                           MLE   Df Model:                            3
Date:                Tue, 27 Oct 2026   Pseudo R-squ.:                  0.1273
Time:                        14:34:30   Log-Likelihood:                -651.57
converged:                       True   LL-Null:                       -746.64
Covariance Type:            nonrobust   LLR p-value:                 5.694e-41
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
const         -2.0310

In [3]:
import numpy as np
df = pd.read_csv("../data/USGS_RG_PFDF_Inventory_v1.csv").replace(-9999, np.nan)
raw = df[df["PreviousUSGSInventory"] == "Staley and others (2016)"]
print("all 2016 rows:\n", raw["Response"].value_counts())
print("rows missing PeakI15:\n", raw[raw["PeakI15"].isna()]["Response"].value_counts())
print(raw[raw["PeakI15"].isna()]["FireName"].value_counts().head())


all 2016 rows:
 Response
0    1101
1     334
Name: count, dtype: int64
rows missing PeakI15:
 Response
0    115
Name: count, dtype: int64
FireName
Coal Seam    108
Harris         5
Canyon         2
Name: count, dtype: int64


In [4]:
import numpy as np
import pandas as pd
import statsmodels.api as sm

df = pd.read_csv("../data/USGS_RG_PFDF_Inventory_v1.csv").replace(-9999, np.nan)
s16 = df[df["PreviousUSGSInventory"] == "Staley and others (2016)"]

published = {
    15: [-3.63, 0.41, 0.67, 0.70],
    30: [-3.61, 0.26, 0.39, 0.50],
    60: [-3.21, 0.17, 0.20, 0.22],
}

for minutes, pub in published.items():
    col = f"PeakI{minutes}"
    x = s16.dropna(subset=[col, "HM23", "dNBR/1000", "KF"]).copy()
    R = x[col] * (minutes / 60)   # mm/hr -> mm over that duration
    X = pd.DataFrame({"TR": x["HM23"] * R, "FR": x["dNBR/1000"] * R, "SR": x["KF"] * R})
    result = sm.Logit(x["Response"], sm.add_constant(X)).fit(disp=0)
    print(minutes, "minute model,", len(x), "rows")
    print(pd.DataFrame({"published": pub, "mine": result.params.values},
                       index=["const", "TR", "FR", "SR"]).round(2))
    print()

15 minute model, 1320 rows
       published  mine
const      -3.63 -2.03
TR          0.41  0.31
FR          0.67 -0.07
SR          0.70  0.09

30 minute model, 1345 rows
       published  mine
const      -3.61 -2.15
TR          0.26  0.24
FR          0.39 -0.08
SR          0.50  0.08

60 minute model, 1351 rows
       published  mine
const      -3.21 -2.41
TR          0.17  0.17
FR          0.20 -0.02
SR          0.22  0.09



In [5]:
raw_cols = ["I15", "T", "F", "S"]
print(d[d["source"] == "Staley and others (2016)"][raw_cols].corr().round(2))

      I15     T     F     S
I15  1.00  0.05  0.18  0.07
T    0.05  1.00  0.39  0.15
F    0.18  0.39  1.00  0.11
S    0.07  0.15  0.11  1.00
